# Exercise 6 - AI Agents

_By Abigail Hayes_

In this exercise we will explore using LLMs as agents. We will focus on open-weight models that can be run locally (if you have a GPU available).

Note that the models we will use in this exercise are significantly larger (>1B parameters) than, e.g., distilbert (67M parameters). <br>
**This means that you will likely need to run this notebook on the BWUniCluster3.0 or on Google Colab to have enough GPU memory and compute.**

This exercise consists of the following parts:
1. Agent setup
2. Building a tool
3. Memory

## Setup

Make sure to select the same kernel as last week to have the previously added modules. We also need the following modules:
* smolagents - for handling agent implementation

In [ ]:
%pip install 'smolagents[toolkit, transformers]' 'flash-linear-attention'

# 1. Agent setup

Setup based on the [smolagents documentation](https://huggingface.co/docs/smolagents/index).

First we will set up a basic coding agent, using the same huggingface model as last week. It is also possible to [access other types of models](https://huggingface.co/docs/smolagents/examples/using_different_models).


In [1]:
from smolagents import CodeAgent, TransformersModel

model = TransformersModel(model_id="Qwen/Qwen3.5-4B")

agent = CodeAgent(tools=[], model=model)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

We have not yet provided the model with any tools, but it will instead write and execute code itself.

In [2]:
result = agent.run("Calculate the sum of numbers from 1 to 10")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Calculate the sum of numbers from 1 to 10                                                                       │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  total_sum = sum(range(1, 11))                                                                                    
  print(f"The sum of numbers from 1 to 10 is: {total_sum}")                                                        
  final_answer(total_sum)                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
The sum of numbers from 1 to 10 is: 55

Final answer: 55

[Step 1: Duration 13.73 seconds| Input tokens: 2,143 | Output tokens: 248]

55


Within the 'toolkit', we already have access to some tools that have been created for us. Let's try allowing the agent to search the internet.

In [3]:
from smolagents import CodeAgent, TransformersModel, DuckDuckGoSearchTool

model = TransformersModel(model_id="Qwen/Qwen3.5-4B")

agent = CodeAgent(tools=[DuckDuckGoSearchTool()], 
                  model=model)

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [4]:
result = agent.run("What is the current weather in Paris?")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the current weather in Paris?                                                                           │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  weather_info = web_search(query="current weather in Paris")                                                      
  print(weather_info)                                                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[Today's Weather in Paris - Hourly Forecast and 
Conditions](https://www.easeweather.com/europe/france/ile-de-france/paris/today)
Get the latest hourly weather updates for Paris today. Detailed forecast including temperature, wind, rain, snow, 
and UV index. Stay informed about today's weather conditions in Paris.

[Paris, Ville de Paris, France Hourly Weather | 
AccuWeather](https://www.accuweather.com/en/fr/paris/75042/hourly-weather-forecast/179181_pc)
Hourly weather forecast in Paris, Ville de Paris, France. Check current conditions in Paris, Ville de Paris, France
with radar, hourly, and more.

[Weather Paris - meteoblue](https://www.meteoblue.com/en/weather/week/paris_france_2988507)
Today's and tonight's professional weather forecast for Paris. Precipitation radar, HD satellite images, and 
current weather warnings, hourly temperature, chance of rain, and sunshine hours.

[Weather Forecast and Conditions for Paris, France - The Weather Channel 
...](https://weather.com/fr/city/paris/today)
Today's and tonight's Paris, France weather forecast, weather conditions and Doppler radar from The Weather Channel
and Weather.com

[Paris, IDF, FR Current Weather - The Weather 
Network](https://www.theweathernetwork.com/en/city/fr/ile-de-france/paris/current)
Get Paris, IDF, FR current weather report with temperature, feels like, wind, humidity, pressure, UV and more from 
TheWeatherNetwork.com.

[Weather forecast and conditions for Paris, France - The Weather 
Channel](https://weather.com/en-SS/fr/city/paris/today)
Today's and tonight's Paris, France weather forecast, weather conditions and Doppler radar from The Weather Channel
and Weather.com

[Weather Paris, France - 14 Days - ️ Worldmeteo](https://www.worldmeteo.info/en/europe/france/paris/weather-100001/)
Weather and weather forecast for Paris, France. ☀️🌧️ Worldwide current weather and 14-day weather forecast

[Paris Weather Today: Live Rain Radar, Hourly & 14-Day Forecast](https://www.weathernow.app/weather/france/paris/)
Check Paris weather today with live rain radar, hourly forecast, 14-day outlook and current conditions in France.

[Paris Weather Forecast](https://www.weather-forecast.com/locations/Paris/forecasts/latest)
The Long-range 12 day forecast also includes detail for Paris weather today. Live weather reports from Paris 
weather stations and weather warnings that include risk of thunder, high UV index and forecast gales. See the links
below the 12-day Paris weather forecast table for other cities and towns nearby along with weather conditions for 
local outdoor activities.

[Today's Weather in Île-de-France - Hourly Forecast and 
Conditions](https://www.easeweather.com/europe/france/ile-de-france/today)
Get the latest hourly weather updates for Île-de-France today. Detailed forecast including temperature, wind, rain,
snow, and UV index. Stay informed about today's weather conditions in Île-de-France.

Out: None

[Step 1: Duration 6.71 seconds| Input tokens: 2,195 | Output tokens: 128]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 12 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Paris, France: 13°C (55°F), Sunny")                                                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Paris, France: 13°C (55°F), Sunny

[Step 12: Duration 14.79 seconds| Input tokens: 76,508 | Output tokens: 2,827]

Paris, France: 13°C (55°F), Sunny


## 2. Building a tool

In fact, we can give the agent access to web search by enabling access to all 'base_tools'. For a code agent, this adds the DuckDuckGo web search and speech-to-text capabilities.

In [1]:
from smolagents import CodeAgent, TransformersModel

model = TransformersModel(model_id="Qwen/Qwen3.5-4B")

agent = CodeAgent(tools=[], 
                  model=model,
                 add_base_tools=True)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [4]:
result = agent.run("Who is the president of Germany?")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Who is the president of Germany?                                                                                │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  president_info = web_search(query="current president of Germany")                                                
  print(president_info)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
## Search Results

[President of Germany - Wikipedia](https://en.wikipedia.org/wiki/President_of_Germany)
The president of Germany, officially titled the federal president of the Federal Republic of Germany (German: 
Bundespräsident der Bundesrepublik Deutschland), [2] is the head of state of Germany. The current officeholder is 
Frank-Walter Steinmeier who was elected on 12 February 2017 and re-elected on 13 February 2022.

[Frank-Walter Steinmeier - Wikipedia](https://en.wikipedia.org/wiki/Frank-Walter_Steinmeier)
Frank-Walter Steinmeier (German: [ˈfʁaŋkˌvaltɐ ˈʃtaɪnˌmaɪ.ɐ] ⓘ; born 5 January 1956) [1] is a German politician who
has served as President of Germany since 2017. [2] He was previously Minister for Foreign Affairs from 2005 to 2009
and again from 2013 to 2017, as well as Vice-Chancellor of Germany from 2007 to 2009.

[Current content on the activities of the Federal 
President](https://www.bundespraesident.de/EN/home/home_node.html)
At the ceremony marking the Day of German Unity in Bremen, Federal President Steinmeier recalled the joyful moments
of reunification and the shared effort of rebuilding. He warned against extremism and social division, and called 
for courage, solidarity, and renewal.

[Frank-Walter Steinmeier（President of 
Germany）_Baiduwiki](https://baike.baidu.com/en/item/Frank-Walter+Steinmeier/3437235)
Frank-Walter Steinmeier (German: Frank-Walter Steinmeier) was born on January 5, 1956, in Detmold, North 
Rhine-Westphalia, Germany. He holds a Doctor of Jurisprudence from the University of Giessen and is a prominent 
figure within the Social Democratic Party of Germany (SPD). He previously served as the Federal Minister for 
Foreign Affairs of Germany. He is the current President of Germany.

[President of Germany - Wikiwand](https://www.wikiwand.com/en/President_of_Germany)
The president of Germany, officially titled the federal president of the Federal Republic of Germany, is the head 
of state of Germany. The current officeholder is Frank-Walter Steinmeier who was elected on 12 February 2017 and 
re-elected on 13 February 2022. He is currently serving his second five-year-term, which began on 19 March 2022.

[Who governs in Germany? |](https://traveltogermanyguide.com/who-governs-in-germany/)
The current Federal President of Germany is Frank-Walter Steinmeier. The Federal President is the ceremonial head 
of state and represents the country in matters of international diplomacy.

[Germany - Federal Foreign Office - Auswärtiges 
Amt](https://www.auswaertiges-amt.de/en/aussenpolitik/germany-230318)
Name: Federal Republic of Germany System of government: Democratic parliamentary federal state Head of State: 
Federal President Frank-Walter Steinmeier, since March 2017 Representative of the Head ...

[What is the name of the president of Germany? 
|](http://traveltogermanyguide.com/what-is-the-name-of-the-president-of-germany/)
The President is elected by a special assembly called the Federal Convention for a term of five years. The current 
President of Germany is Frank-Walter Steinmeier. He has been in office since March 19, 2017. As the head of state, 
the President of Germany represents the country internationally and performs ceremonial duties within Germany.

[Der Bundespräsident - Frank-Walter 
Steinmeier](https://www.bundespraesident.de/EN/federal-president/frank-walter-steinmeier_node.html)
Learn more about the current president of Germany, his biography, speeches, engagements and focal areas of his term
of office. Find out how he promotes dialogue, cohesion and transformation in Germany and beyond.

[Frank-Walter Steinmeier | Facts & Biography | 
Britannica](https://www.britannica.com/biography/Frank-Walter-Steinmeier)
Frank-Walter Steinmeier (born January 5, 1956, Detmold, West Germany) is a German Social Democratic Party 
(Sozialdemokratische Partei Deutschlands; SPD) politician who in the early 21st century served as vice-chancellor 
(2007-09) and foreign minister (2005-0

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Frank-Walter Steinmeier")                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Frank-Walter Steinmeier

[Step 2: Duration 3.80 seconds| Input tokens: 5,706 | Output tokens: 240]

Frank-Walter Steinmeier


We can also manually call tools, to see what information they would provide to the agent.

In [5]:
from smolagents import WebSearchTool

search_tool = WebSearchTool()
print(search_tool("Who's the current president of Germany?"))

## Search Results

[President of Germany - Wikipedia](https://en.wikipedia.org/wiki/President_of_Germany)
 The  president  of  Germany , officially titled the federal president  of  the Federal Republic of Germany (German: Bundespräsident der Bundesrepublik Deutschland), [2] is the head of state of Germany . The  current officeholder is Frank-Walter Steinmeier who was elected on 12 February 2017 and re-elected on 13 February 2022.

[Frank-Walter Steinmeier - Wikipedia](https://en.wikipedia.org/wiki/Frank-Walter_Steinmeier)
Frank-Walter Steinmeier (German: [ˈfʁaŋkˌvaltɐ ˈʃtaɪnˌmaɪ.ɐ] ⓘ; born 5 January 1956) [1] is a German politician who has served as President  of  Germany since 2017. [2] He was previously Minister for Foreign Affairs from 2005 to 2009 and again from 2013 to 2017, as well as Vice-Chancellor of Germany from 2007 to 2009.

[Der Bundespräsident - Frank-Walter Steinmeier](https://www.bundespraesident.de/EN/federal-president/frank-walter-steinmeier_node.html)
Federal Presid

In previous weeks, we have learnt basic text analysis methods. Let us now give the agent access to some tools that would help it to analyse text.

### Text statistics

_Text from: [www.bundespraesident.de](https://www.bundespraesident.de/SharedDocs/Reden/EN/Frank-Walter-Steinmeier/Reden/2026/261008-State-banquet-Paris.html)_

In [6]:
from smolagents import tool

@tool
def text_stats(text: str) -> dict:
    """Analyse a text and return basic statistics.

    Args:
        text: The text to analyse.

    Returns:
        A dictionary containing character, word, and line counts.
    """
    return {
        "characters": len(text),
        "words": len(text.split()),
        "lines": len(text.splitlines()),
    }


In [7]:
from smolagents import CodeAgent, TransformersModel

model = TransformersModel(model_id="Qwen/Qwen3.5-4B")

agent = CodeAgent(tools=[text_stats], 
                  model=model,
                 add_base_tools=True)

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [10]:
result = agent.run("What are the basis statistics of the following text? 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe, and will miss you', the Federal President said.'")
print(result)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What are the basis statistics of the following text? 'At the state banquet in Paris, Federal President          │
│ Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all firmly believe in the  │
│ need for this shared Europe - and we all profoundly appreciate what you do for Europe, and will miss you', the  │
│ Federal President said.'                                                                                        │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  text = 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel    
  Macron as a great European. "We all firmly believe in the need for this shared Europe - and we all profoundly    
  appreciate what you do for Europe, and will miss you", the Federal President said.'                              
  stats = text_stats(text=text)                                                                                    
  print(stats)                                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'characters': 294, 'words': 50, 'lines': 1}

Out: None

[Step 1: Duration 9.50 seconds| Input tokens: 2,353 | Output tokens: 221]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer({'characters': 294, 'words': 50, 'lines': 1})                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'characters': 294, 'words': 50, 'lines': 1}

[Step 2: Duration 5.19 seconds| Input tokens: 4,994 | Output tokens: 358]

{'characters': 294, 'words': 50, 'lines': 1}


But if we ask a more specific question, where the result can be extracted from our tool, the agent may need specific guidance to know to use it.

In [11]:
result = agent.run("How many words are there in the following text? 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe, and will miss you', the Federal President said.'")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How many words are there in the following text? 'At the state banquet in Paris, Federal President Steinmeier    │
│ paid tribute to French President Emmanuel Macron as a great European. 'We all firmly believe in the need for    │
│ this shared Europe - and we all profoundly appreciate what you do for Europe, and will miss you', the Federal   │
│ President said.'                                                                                                │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  text = 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel    
  Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly    
  appreciate what you do for Europe, and will miss you', the Federal President said.'                              
                                                                                                                   
  # Split the text by whitespace and count the words                                                               
  words = text.split()                                                                                             
  word_count = len(words)                                                                                          
                                                                                                                   
  print(f"Number of words: {word_count}")                                                                          
  print(f"Words: {words}")                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code parsing failed on line 1 due to: SyntaxError: invalid syntax (<unknown>, line 1)
text = 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel 
Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly 
appreciate what you do for Europe, and will miss you', the Federal President said.'
                                                                                                                   
                         ^

[Step 1: Duration 11.63 seconds| Input tokens: 2,353 | Output tokens: 331]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(50)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 50

[Step 3: Duration 3.55 seconds| Input tokens: 8,453 | Output tokens: 698]

50


With a small hint, the agent can use the provided function.

In [12]:
result = agent.run("How many words are there in the following text (use the text statistics)? 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe, and will miss you', the Federal President said.'")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How many words are there in the following text (use the text statistics)? 'At the state banquet in Paris,       │
│ Federal President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all      │
│ firmly believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe,    │
│ and will miss you', the Federal President said.'                                                                │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  text = 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel    
  Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly    
  appreciate what you do for Europe, and will miss you', the Federal President said.'                              
  stats = text_stats(text=text)                                                                                    
  print(stats)                                                                                                     
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code parsing failed on line 1 due to: SyntaxError: invalid syntax (<unknown>, line 1)
text = 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel 
Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly 
appreciate what you do for Europe, and will miss you', the Federal President said.'
                                                                                                                   
                         ^

[Step 1: Duration 10.92 seconds| Input tokens: 2,358 | Output tokens: 282]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(50)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 50

[Step 3: Duration 4.73 seconds| Input tokens: 8,128 | Output tokens: 572]

50


Alternatively, we can try to improve the tool description, so that the agent realises how it can be used.

In [16]:
from smolagents import tool

@tool
def text_stats(text: str) -> dict:
    """Use to return the number of characters, number of words and number of lines in a text.

    Args:
        text: The text to analyse.

    Returns:
        A dictionary containing character, word, and line counts.
    """
    return {
        "characters": len(text),
        "words": len(text.split()),
        "lines": len(text.splitlines()),
    }

In [17]:
agent = CodeAgent(tools=[text_stats], 
                  model=model,
                 add_base_tools=True)

result = agent.run("How many words are there in the following text? 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe, and will miss you', the Federal President said.'")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How many words are there in the following text? 'At the state banquet in Paris, Federal President Steinmeier    │
│ paid tribute to French President Emmanuel Macron as a great European. 'We all firmly believe in the need for    │
│ this shared Europe - and we all profoundly appreciate what you do for Europe, and will miss you', the Federal   │
│ President said.'                                                                                                │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  text = 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel    
  Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly    
  appreciate what you do for Europe, and will miss you', the Federal President said.'                              
  result = text_stats(text=text)                                                                                   
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code parsing failed on line 1 due to: SyntaxError: invalid syntax (<unknown>, line 1)
text = 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel 
Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly 
appreciate what you do for Europe, and will miss you', the Federal President said.'
                                                                                                                   
                         ^

[Step 1: Duration 18.03 seconds| Input tokens: 2,363 | Output tokens: 492]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(50)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 50

[Step 3: Duration 2.91 seconds| Input tokens: 8,205 | Output tokens: 806]

50


## Task 1

We will do some further analysis of the text.

1. Ask the agent to count how many times the letter 'f' appears in the text.
2. Build a tool which counts how often each letter appears in the text.
3. Get the agent to use this tool to find out how many times the letter 'f' appears in the text.
4. How do the two responses differ? What might be the advantages or disadvantages of using tools for this task?


In [18]:
result = agent.run("How many times does the letter 'f' appear in the following text? 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe, and will miss you', the Federal President said.'")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How many times does the letter 'f' appear in the following text? 'At the state banquet in Paris, Federal        │
│ President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all firmly       │
│ believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe, and will  │
│ miss you', the Federal President said.'                                                                         │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  text = "At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel    
  Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly    
  appreciate what you do for Europe, and will miss you', the Federal President said."                              
                                                                                                                   
  # Count occurrences of 'f' and 'F'                                                                               
  count_f = text.lower().count('f')                                                                                
  print(f"The letter 'f' appears {count_f} times in the text.")                                                    
  final_answer(count_f)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
The letter 'f' appears 7 times in the text.

Final answer: 7

[Step 1: Duration 24.78 seconds| Input tokens: 2,368 | Output tokens: 686]

7


In [19]:
from collections import Counter

@tool
def letter_frequency(text: str) -> dict:
    """Count how often each letter appears in a text.

    Use this tool when the user asks for letter frequencies
    or wants to know how often individual letters occur.

    Counting is case-insensitive. Ignore spaces, digits,
    punctuation, and other non-letter characters.

    Args:
        text: The text to analyse.

    Returns:
        A dictionary mapping each letter to its occurrence count.
    """
    counts = Counter(
        char.lower()
        for char in text
        if char.isalpha()
    )

    return dict(sorted(counts.items()))

In [20]:
agent = CodeAgent(tools=[letter_frequency], 
                  model=model,
                 add_base_tools=True)

result = agent.run("How many times does the letter 'f' appear in the following text? 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe, and will miss you', the Federal President said.'")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How many times does the letter 'f' appear in the following text? 'At the state banquet in Paris, Federal        │
│ President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all firmly       │
│ believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe, and will  │
│ miss you', the Federal President said.'                                                                         │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  text = "At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel    
  Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly    
  appreciate what you do for Europe, and will miss you', the Federal President said."                              
                                                                                                                   
  frequency = letter_frequency(text=text)                                                                          
  print(frequency)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'a': 22, 'b': 3, 'c': 3, 'd': 13, 'e': 39, 'f': 7, 'g': 1, 'h': 7, 'i': 17, 'l': 12, 'm': 6, 'n': 15, 'o': 12, 
'p': 11, 'q': 1, 'r': 20, 's': 12, 't': 18, 'u': 9, 'v': 1, 'w': 4, 'y': 4}

Out: None

[Step 1: Duration 10.63 seconds| Input tokens: 2,407 | Output tokens: 258]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(7)                                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 7

[Step 2: Duration 9.28 seconds| Input tokens: 5,235 | Output tokens: 525]

7


In [21]:
result = agent.run("How many times do the letters 'f', 'g' and 'h' appear in the following text? 'At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe, and will miss you', the Federal President said.'")
print(result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How many times do the letters 'f', 'g' and 'h' appear in the following text? 'At the state banquet in Paris,    │
│ Federal President Steinmeier paid tribute to French President Emmanuel Macron as a great European. 'We all      │
│ firmly believe in the need for this shared Europe - and we all profoundly appreciate what you do for Europe,    │
│ and will miss you', the Federal President said.'                                                                │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  text = "At the state banquet in Paris, Federal President Steinmeier paid tribute to French President Emmanuel    
  Macron as a great European. 'We all firmly believe in the need for this shared Europe - and we all profoundly    
  appreciate what you do for Europe, and will miss you', the Federal President said."                              
                                                                                                                   
  frequencies = letter_frequency(text)                                                                             
  print(frequencies)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
{'a': 22, 'b': 3, 'c': 3, 'd': 13, 'e': 39, 'f': 7, 'g': 1, 'h': 7, 'i': 17, 'l': 12, 'm': 6, 'n': 15, 'o': 12, 
'p': 11, 'q': 1, 'r': 20, 's': 12, 't': 18, 'u': 9, 'v': 1, 'w': 4, 'y': 4}

Out: None

[Step 1: Duration 10.13 seconds| Input tokens: 2,414 | Output tokens: 290]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(15)                                                                                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 15

[Step 3: Duration 5.04 seconds| Input tokens: 8,308 | Output tokens: 646]

15


## 3. Memory

We also generally want agents to be able to work with a memory. Since smolagents aims to be lightweight, the memory system is lightweight and only persists through a single call. Other frameworks such as Autogen or Langchain have more elaborate memory systems as default. However, we can instead use a tool for the purpose.

In [2]:
from MemoryBank import MemoryBankTool

memory_bank_tool = MemoryBankTool(
    memory_bank_dir_path=f"./memory_bank",
)

memory_bank_instruction = """ Use the Memory Bank tool to keep track of your work. You maintain a structured Memory Bank to ensure continuity and coherence across your problem-solving steps:
  1. Project Overview: Core objectives, scope, and user-defined goals for the current task.
  2. Active Progress Tracking: Current tasks, recent changes, and next steps.
  Remember to keep them updated. """

agent = CodeAgent(tools=[memory_bank_tool], 
                  model=model,
                 add_base_tools=True,
                 instructions=[memory_bank_instruction])

2026-10-12 17:09:41,128 - INFO - Created default Memory Bank section: Project Overview
2026-10-12 17:09:41,133 - INFO - Created default Memory Bank section: Active Progress Tracking
2026-10-12 17:09:41,133 - INFO - MemoryBankTool initialized with directory: /pfs/data6/home/ma/ma_ma/ma_ahayes/memory_bank


In [3]:
request = """
Do a deep research about vaccines, remember to use your Memory Bank:
- find the 5 most common vaccines used throughout the world nowadays
- find source material about the safety profile of each one of these vaccines
- find recommendations of application of these vaccines for public authorities, including schedules, target groups and other
similar relevant information
- add some historical context about each vaccine, including the year of introduction and the disease it is meant to prevent
- produce a final report with the relevant content summarized in a single markdown document, with references cited where needed
"""

response = agent.run(request)
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Do a deep research about vaccines, remember to use your Memory Bank:                                            │
│ - find the 5 most common vaccines used throughout the world nowadays                                            │
│ - find source material about the safety profile of each one of these vaccines                                   │
│ - find recommendations of application of these vaccines for public authorities, including schedules, target     │
│ groups and other                                                                                                │
│ similar relevant information                                                                                    │
│ - add some historical context about each vaccine, including the year of introduction and the disease it is      │
│ meant to prevent                                                                                                │
│ - produce a final report with the relevant content summarized in a single markdown document, with references    │
│ cited where needed                                                                                              │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  memory_bank(action="update", section="Project Overview", content="# Vaccine Research Project\n\n##               
  Objectives\n- Identify the 5 most common vaccines used worldwide\n- Research safety profiles for each            
  vaccine\n- Find application recommendations (schedules, target groups)\n- Gather historical context (year of     
  introduction, disease prevented)\n- Produce a comprehensive markdown report with references\n\n## Current        
  Status\n- [ ] Research most common vaccines\n- [ ] Research safety profiles\n- [ ] Research application          
  recommendations\n- [ ] Research historical context\n- [ ] Compile final report")                                 
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out: Successfully updated Memory Bank section: Project Overview

...

Execution logs:
## Search Results

[Rotavirus Vaccination Coverage and Adherence to the Advisory Committee 
...](https://www.cdc.gov/mmwr/preview/mmwrhtml/mm5715a2.htm)
In addition, health-care providers should remain vigilant in following the ACIP-recommended vaccination schedule 
for rotavirus vaccine and are reminded to report any adverse events to the Vaccine Adverse Events Reporting System.
Acknowledgments

[Safety profile of rotavirus vaccines among individuals aged ≥8 months 
...](https://www.sciencedirect.com/science/article/pii/S0264410X20314663)
Abstract Introduction In 2006 and 2008, two live, oral rotavirus vaccines, RotaTeq (RV5) and Rotarix (RV1), were 
introduced into the routine immunization program in the United States. A previous rotavirus vaccine, RotaShield, 
was associated with an increased risk of intussusception, with data suggesting an age-dependent variation in risk.

[Prevention of rotavirus gastroenteritis among infants and children ...](https://pubmed.ncbi.nlm.nih.gov/16902398/)
In February 2006, a live, oral, human-bovine reassortant rotavirus vaccine (RotaTeq) was licensed for use among 
U.S. infants. The Advisory Committee on Immunization Practices recommends routine vaccination of U.S. infants with 
3 doses of this rotavirus vaccine administered orally at ages 2, 4, and 6 months. The first dose should be 
administered between ages 6-12 weeks. Subsequent doses should ...

[Chapter 19: Rotavirus | Pink Book | 
CDC](https://www.cdc.gov/pinkbook/hcp/table-of-contents/chapter-19-rotavirus.html)
The marked reduction in rotavirus disease burden in the United States following the introduction of rotavirus 
vaccine in 2006 has been documented by data on hospitalizations and emergency department care for diarrhea among 
young children.

[Rotavirus Vaccine Story: From Discovery to the Eventual Control of 
...](https://academic.oup.com/jid/article/224/Supplement_4/S331/6378096)
Abstract Worldwide, rotavirus is the leading pathogen causing severe diarrhea in children and a major cause of 
under 5 years mortality. In 1998, the first rotavirus vaccine, RotaShield, was licensed in the United States but a 
rare adverse event, intussusception, led to its withdrawal. Seven years passed before the next generation of 
vaccines became available, Rotarix (GSK) and Rotateq (Merck ...

[PDF A decade of experience with rotavirus vaccination in the United States 
...](https://stacks.cdc.gov/view/cdc/119270/cdc_119270_DS1.pdf)
Abstract Introduction: Prior to 2006, nearly every U.S. child was infected with rotavirus by 5 years of age, and 
rotavirus was the leading cause of severe childhood gastroenteritis. In February 2006 and June 2008, the Advisory 
Committee on Immunization Practices recommended a live attenuated pentavalent rotavirus vaccine (RV5) and a 
monovalent rotavirus vaccine (RV1), respectively, for routine ...

[Global rotavirus vaccine introductions and coverage: 2006 - 
2016](https://pmc.ncbi.nlm.nih.gov/articles/PMC6183203/)
This review compares rotavirus vaccine (RV) introductions and vaccine coverage by region, country income status and
Gavi-eligibility from 2006-2016. Gross National Income data from the World Bank and surviving infant population 
from United Nations Population Division was obtained for 2016.

[Safety and Efficacy of an Attenuated Vaccine against Severe Rotavirus 
...](https://www.nejm.org/doi/full/10.1056/NEJMoa052434)
The safety and efficacy of an attenuated G1P [8] human rotavirus (HRV) vaccine were tested in a randomized, 
double-blind, phase 3 trial.

[Safety profile of rotavirus vaccines among individuals aged ≥8 months 
...](https://www.researchgate.net/publication/347233724_Safety_profile_of_rotavirus_vaccines_among_individuals_aged
_8_months_of_age_United_States_vaccine_adverse_event_reporting_system_VAERS_2006-2019)
Request PDF | Safety profile of rotavirus vaccines among individuals aged ≥8 months of age, United States, vaccine 
adverse event reporting system (VAERS), 2006-2019

[Step 20: Duration 5.94 seconds| Input tokens: 339,456 | Output tokens: 2,496]

Reached max steps.

[Step 21: Duration 96.14 seconds| Input tokens: 370,389 | Output tokens: 5,217]

The user wants me to create a comprehensive research report about vaccines. Based on my research, I need to:

1. Identify the 5 most common vaccines worldwide
2. Research safety profiles for each
3. Find application recommendations (schedules, target groups)
4. Add historical context (year of introduction, disease prevented)
5. Create a final markdown report with references

From my research, the 5 most common vaccines worldwide are:
1. DTP (Diphtheria-Tetanus-Pertussis)
2. Polio Vaccine (IPV/OPV)
3. BCG (Bacille Calmette-Guérin)
4. Measles Vaccine
5. Rotavirus Vaccine

I have gathered information about each of these vaccines from various sources. Now I need to compile this into a comprehensive markdown report with proper citations.

Let me create the final report based on the information I've gathered.
</think>

# Comprehensive Vaccine Research Report

## Executive Summary

This report provides a detailed analysis of the five most commonly used vaccines worldwide, including their safe

In [4]:
with open(f"./memory_bank/vaccine_report.md", "w") as f:
    f.write(response)

We can continue to reuse the information that the agent has stored.

In [5]:
request = """
Using only the information stored in your memory, when was the TB vaccine first developed?"""

response = agent.run(request)
print(response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Using only the information stored in your memory, when was the TB vaccine first developed?                      │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen3.5-4B ───────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  memory_list = memory_bank(action="list", section="Project Overview")                                             
  print(memory_list)                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Available Memory Bank sections:
- Active Progress Tracking
- Project Overview
- Vaccine Report

Out: None

[Step 1: Duration 7.62 seconds| Input tokens: 2,580 | Output tokens: 216]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 7 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The TB vaccine (BCG - Bacille Calmette-Guérin) was first developed in 1921.")                      
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The TB vaccine (BCG - Bacille Calmette-Guérin) was first developed in 1921.

[Step 7: Duration 3.96 seconds| Input tokens: 62,463 | Output tokens: 1,519]

The TB vaccine (BCG - Bacille Calmette-Guérin) was first developed in 1921.


## Task 2

Choose a topic from the options below for your agent to research. Then ask it to answer the accompanying questions from it's memory.

### Wrestling at the 1924 Olympics
1. Which country won the most wrestling medals at the 1924 Olympics?
2. How many wrestling events were there for women at the 1924 Olympics?
3. Where did you find the information about wrestling at the 1924 Olympics?

### Kazakhstan
1. Why was the capital city of Kazakhstan changed?
2. Where did you find out information about Kazakhstan?
3. What is the most important information for me to know about Kazakhstan?